# 5.12 改幾個字就算新資料嗎？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：先分整份文件，再切小窗口**

近似改寫的同一家族也放在同一側。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/split.svg")))

**先想一想：**只改句子末尾一個字，相似度應完全變0嗎？

近似重複不是hash相同。把字元片段當集合，量兩段文字共有多少片段，再以家族為單位切分，避免改寫版本跨split。

**把直覺寫成數學：**Jaccard(A,B)=|A∩B|/|A∪B|；這是相似度啟發式，不是語意等價證明。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
def grams(s, n=2):
    return {s[i : i + n] for i in range(len(s) - n + 1)}


a, b = grams("紅色圓形在左邊"), grams("紅色圓形在右邊")
score = len(a & b) / len(a | b)
print(score, a & b)

**如何讀結果：**閾值取決於資料；短句少量片段容易誤判，應人工抽查。

**只改一件事：**只把片段長度由2改爲3。
